# Theme 1: Bug-Fix Adoption Trends

**RQ1:** How does AI bug-fixing *volume* change over time?  
**RQ7:** Do developers *switch agents* for bug fixing over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)  
Scope: closed bug-fix PRs only (Copilot, Cursor, Claude Code, Devin + Human baseline)

**Methodology notes (applied to every figure below):**
- **Survivorship cutoff:** PRs created within the last 30 days of coverage are dropped — because we only see *closed* PRs, the most recent weeks are biased toward fast-decision PRs. `load_fix_prs` applies this automatically.
- **Model-version drift:** a single named agent spans several underlying models over 15 months, so within-agent trends mix product evolution with usage shift. Known model-release dates are annotated as grey vertical lines (see Claude Code).

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\CLI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    odds_ratio_ci, cliffs_delta, bh_correct,
    wilson_ci, bootstrap_median_ci, annotate_model_releases,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
    MIN_N_PROP, MIN_N_MEDIAN,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()
print('Agent fix PRs:', f"{len(agents_df):,}")
print('Human fix PRs:', f"{len(human_df):,}")

Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Fix PRs loaded: 371,577  |  Agent: 108,080  |  Human: 263,497


Agent fix PRs: 108,080
Human fix PRs: 263,497


## RQ1: How does AI bug-fixing volume change over time?

In [4]:
# Monthly volume: agent (all 4) vs human
monthly = df.groupby(['month', 'is_agent']).size().unstack(fill_value=0)
monthly.columns = ['Human', 'Agent']
monthly.index   = monthly.index.astype(str)
monthly['Total']    = monthly['Agent'] + monthly['Human']
monthly['Agent_%']  = (monthly['Agent'] / monthly['Total'] * 100).round(1)
print('Monthly volume (Agent vs Human):')
print(monthly.to_string())

Monthly volume (Agent vs Human):
         Human  Agent  Total  Agent_%
month                                
2024-12  14240   2924  17164     17.0
2025-01  16317   3158  19475     16.2
2025-02  17312   3840  21152     18.2
2025-03  19081   4854  23935     20.3
2025-04  19050   5041  24091     20.9
2025-05  18510   7459  25969     28.7
2025-06  18992  12729  31721     40.1
2025-07  21451  25354  46805     54.2
2025-08  18648  10802  29450     36.7
2025-09  20021   7123  27144     26.2
2025-10  20691   6396  27087     23.6
2025-11  19074   6154  25228     24.4
2025-12  19285   5816  25101     23.2
2026-01  20825   6430  27255     23.6


In [5]:
# Figure: monthly volume — agent vs human (line chart)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly.index, monthly['Agent'], 'o-', color='#1f77b4', label='Agent (all)')
ax.plot(monthly.index, monthly['Human'], 's--', color='#7f7f7f', label='Human', alpha=0.7)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff (Jul-25)')
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix PR Volume — Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_agent_vs_human', THEME1_DIR)

  -> Saved: results\theme1_figures\rq1_volume_agent_vs_human.png


WindowsPath('results/theme1_figures/rq1_volume_agent_vs_human.png')

In [6]:
# Monthly volume per agent
per_agent = agents_df.groupby(['month', 'agent']).size().unstack(fill_value=0)
per_agent.index = per_agent.index.astype(str)
cols = [a for a in AGENTS if a in per_agent.columns]
per_agent = per_agent[cols]
print('Monthly volume by agent:')
print(per_agent.to_string())

Monthly volume by agent:
agent    Copilot  Cursor  Claude_Code  Devin
month                                       
2024-12       69    1228         1389    238
2025-01       72    1284         1410    392
2025-02       72    1297         1703    768
2025-03       86    1525         2230   1013
2025-04      101    1400         2181   1359
2025-05     1068    2143         3016   1232
2025-06     2942    3586         5381    820
2025-07     8036    9071         7389    858
2025-08     3272    3313         3853    364
2025-09     1697    2329         2898    199
2025-10     1749    2024         2379    244
2025-11     1951    1834         2219    150
2025-12     1568    1689         2329    230
2026-01     1795    1934         2387    314


In [7]:
# Figure: per-agent stacked bar (monthly volume)
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in per_agent.columns]
per_agent.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.axvline(
    per_agent.index.tolist().index('2025-07') + 0.5,
    color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff'
)
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix Volume by Agent')
ax.legend(loc='upper left')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_by_agent', THEME1_DIR)

  -> Saved: results\theme1_figures\rq1_volume_by_agent.png


WindowsPath('results/theme1_figures/rq1_volume_by_agent.png')

## RQ7: Do developers switch agents for bug fixing over time?

Measured as each agent's **share of total agent bug-fix PRs** per month.

In [8]:
# Agent market share: % of agent bug-fix PRs from each tool per month
share = per_agent.div(per_agent.sum(axis=1), axis=0) * 100
print('Agent market share (%) per month:')
print(share.round(1).to_string())

Agent market share (%) per month:
agent    Copilot  Cursor  Claude_Code  Devin
month                                       
2024-12      2.4    42.0         47.5    8.1
2025-01      2.3    40.7         44.6   12.4
2025-02      1.9    33.8         44.3   20.0
2025-03      1.8    31.4         45.9   20.9
2025-04      2.0    27.8         43.3   27.0
2025-05     14.3    28.7         40.4   16.5
2025-06     23.1    28.2         42.3    6.4
2025-07     31.7    35.8         29.1    3.4
2025-08     30.3    30.7         35.7    3.4
2025-09     23.8    32.7         40.7    2.8
2025-10     27.3    31.6         37.2    3.8
2025-11     31.7    29.8         36.1    2.4
2025-12     27.0    29.0         40.0    4.0
2026-01     27.9    30.1         37.1    4.9


In [9]:
# Figure: market share line chart
fig, ax = plt.subplots(figsize=(13, 5))
for agent in share.columns:
    ax.plot(share.index, share[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=2)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
# Annotate Claude Code model-release boundaries (within-agent product drift)
annotate_model_releases(ax, 'Claude_Code', list(share.index))
ax.set_xlabel('Month')
ax.set_ylabel('Share of Agent Bug-Fix PRs (%)')
ax.set_title('RQ7: Agent Market Share in Bug Fixing Over Time')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_market_share', THEME1_DIR)

  -> Saved: results\theme1_figures\rq7_agent_market_share.png


WindowsPath('results/theme1_figures/rq7_agent_market_share.png')

In [10]:
# Figure: stacked 100% bar — proportional agent usage
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in share.columns]
share.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 100)
ax.set_title('RQ7: Proportional Agent Usage for Bug Fixing (Monthly)')
ax.legend(loc='upper right')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_share_stacked', THEME1_DIR)

  -> Saved: results\theme1_figures\rq7_agent_share_stacked.png


WindowsPath('results/theme1_figures/rq7_agent_share_stacked.png')

## Threats to validity (Theme 1)

- **Survivorship in the tail.** Only closed PRs are observed, so the most recent weeks over-represent fast-decision PRs. We drop the final 30 days, but months near the boundary remain partly affected — read the last 1–2 points cautiously.
- **Volume ≠ adoption.** PR counts reflect both how many developers use an agent *and* how active those repos are. A spike can be one very active repo (see the Jul-25 concentration check in `investigate.py`).
- **Market share is compositional.** RQ7 share is each agent's fraction of *agent* PRs; a rise for one agent mechanically lowers others. It does not measure absolute adoption.
- **Model-version drift.** "Claude_Code" (and every agent) spans multiple underlying models across the window; the annotated release lines mark where within-agent trends conflate product change with usage change.
- **Repo-coverage shift.** The set of repos in the dataset is not fixed month to month; volume/share trends partly track which repos entered coverage. RQ8 (Theme 3) addresses this by restricting to the AIDev repo set.